# Logistic Regression - Finetuned LaBSE on Training Corpus
- LaBSE finetuned on training data only before encoding
- Same stratified 5-fold CV protocol as all other models
- Grid search for best config on validation set
- Final evaluation on fixed 20% holdout test

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, classification_report
)
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import wandb

print('Libraries loaded.')

## 2. WandB Init

In [ ]:
wandb_key = os.environ.get('WANDB_API_KEY')
if not wandb_key:
    raise ValueError('Set WANDB_API_KEY environment variable')
wandb.login(key=wandb_key)

wandb.init(
    project = 'commitment-mining',
    name    = 'ml-logistic-regression-LaBSE-finetuned',
    config  = {'model': 'logistic_regression', 'embedding': 'LaBSE-finetuned'},
    tags    = ['logistic-regression', 'machine-learning', 'LaBSE', 'finetuned']
)

## 3. Load Data

In [ ]:
TRAIN_PATH  = 'Commitment-Mining/dataset/train_80p.xlsx'
TEST_PATH   = 'Commitment-Mining/dataset/test_20p.xlsx'
TEXT_COL    = 'statements (ne)'
LABEL_COL   = 'final_label'
RANDOM_SEED = 42

train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

for df in [train_df, test_df]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

print(f'Train size : {len(train_df)}')
print(f'Test size  : {len(test_df)}')
print('Train label distribution:')
print(train_df[LABEL_COL].value_counts())

wandb.log({
    'train_size' : len(train_df),
    'test_size'  : len(test_df),
    **train_df[LABEL_COL].value_counts().to_dict()
})

## 4. Build Stratification Key

In [ ]:
train_df['sentence_length'] = train_df[TEXT_COL].apply(
    lambda x: pd.cut(
        [len(x.split())],
        bins=[0, 5, 10, 20, 50, 999],
        labels=['xs', 's', 'm', 'l', 'xl']
    )[0]
)

train_df['strat_key'] = (
    train_df['province'].astype(str)           + '_' +
    train_df['sentence_length'].astype(str)    + '_' +
    train_df['district/gaupalika'].astype(str) + '_' +
    train_df[LABEL_COL].astype(str)
)

counts = train_df['strat_key'].value_counts()
rare   = counts[counts < 5].index
train_df['strat_key'] = train_df['strat_key'].apply(
    lambda x: 'rare' if x in rare else x
)

print(f'Unique strat keys : {train_df["strat_key"].nunique()}')

## 5. Encode Labels

In [ ]:
le      = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL].values)
y_test  = le.transform(test_df[LABEL_COL].values)
groups  = train_df['strat_key'].values

X_train = train_df[TEXT_COL].to_numpy(dtype=str)
X_test  = test_df[TEXT_COL].to_numpy(dtype=str)

print(f'Classes : {le.classes_}')
print(f'y_train : {y_train.shape} dtype: {y_train.dtype}')

## 6. Finetune LaBSE on Training Data Only
LaBSE is finetuned on training corpus only.
Test set is never seen during finetuning.
Uses contrastive loss with positive pairs from same class.

In [ ]:
FINETUNE_EPOCHS    = 5
FINETUNE_BATCH     = 16
FINETUNE_LR        = 2e-4
FINETUNED_PATH     = '/home/rupak/Desktop/Commitment-Mining/embeddings/labse_finetuned'

# Load pretrained LaBSE
print('Loading pretrained LaBSE...')
labse = SentenceTransformer('sentence-transformers/LaBSE')

# Build training examples for finetuning
# Use sentence pairs from same class as positive pairs
train_texts  = X_train.tolist()
train_labels = y_train.tolist()

# Create InputExample for each sentence with its label
# CosineSimilarityLoss expects pairs -- we use each sentence with itself
# as a supervised signal via softmax classification loss
train_examples = [
    InputExample(texts=[text, text], label=float(label))
    for text, label in zip(train_texts, train_labels)
]

train_dataloader = DataLoader(
    train_examples,
    shuffle = True,
    batch_size = FINETUNE_BATCH
)

# Use CosineSimilarityLoss for finetuning
train_loss = losses.CosineSimilarityLoss(model=labse)

print(f'Finetuning LaBSE for {FINETUNE_EPOCHS} epochs on {len(train_examples)} training sentences...')
labse.fit(
    train_objectives = [(train_dataloader, train_loss)],
    epochs           = FINETUNE_EPOCHS,
    warmup_steps     = int(len(train_dataloader) * 0.1),
    optimizer_params = {'lr': FINETUNE_LR},
    output_path      = FINETUNED_PATH,
    show_progress_bar= True
)

print(f'Finetuning complete. Model saved to {FINETUNED_PATH}')

wandb.config.update({
    'finetune_epochs'    : FINETUNE_EPOCHS,
    'finetune_batch_size': FINETUNE_BATCH,
    'finetune_lr'        : FINETUNE_LR,
    'finetune_samples'   : len(train_examples)
})

## 7. Encode with Finetuned LaBSE
Embeddings computed once with finetuned model.
Test set encoded here but never used until final evaluation.

In [ ]:
# Load finetuned model
labse_finetuned = SentenceTransformer(FINETUNED_PATH)

print('Encoding with finetuned LaBSE...')
X_train_emb = labse_finetuned.encode(
    X_train.tolist(),
    batch_size        = 32,
    show_progress_bar = True,
    convert_to_numpy  = True
).astype(np.float32)

X_test_emb = labse_finetuned.encode(
    X_test.tolist(),
    batch_size        = 32,
    show_progress_bar = True,
    convert_to_numpy  = True
).astype(np.float32)

print(f'X_train_emb : {X_train_emb.shape} dtype: {X_train_emb.dtype}')
print(f'X_test_emb  : {X_test_emb.shape}  dtype: {X_test_emb.dtype}')

wandb.log({
    'embedding_dim'    : X_train_emb.shape[1],
    'x_train_samples'  : X_train_emb.shape[0],
    'x_test_samples'   : X_test_emb.shape[0]
})

## 8. Grid Search - Find Best Config
Same grid and protocol as pretrained LaBSE-LR for direct comparison.

In [ ]:
param_grid = {
    'C'           : [0.01, 0.1, 1.0, 10.0],
    'penalty'     : ['l1', 'l2'],
    'class_weight': [None, 'balanced']
}

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

grid_search = GridSearchCV(
    estimator  = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, solver='saga'),
    param_grid = param_grid,
    cv         = sgkf,
    scoring    = 'f1_macro',
    n_jobs     = -1,
    verbose    = 1,
    refit      = True
)

grid_search.fit(X_train_emb, y_train, groups=groups)

print('Grid search complete.')
print(f'Best CV macro F1 : {grid_search.best_score_:.4f}')
print('Best params:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')

wandb.config.update({
    'best_cv_f1': grid_search.best_score_,
    **{f'best_param/{k}': v for k, v in grid_search.best_params_.items()}
})

## 9. Per-Fold CV - All 5 Metrics with Best Config

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy'  : accuracy_score(y_true, y_pred),
        'precision' : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'    : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1'        : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'auroc'     : roc_auc_score(y_true, y_proba)
    }

fold_metrics = []

for fold, (tr_idx, val_idx) in enumerate(
    sgkf.split(X_train_emb, y_train, groups=groups)
):
    X_tr,  X_val  = X_train_emb[tr_idx], X_train_emb[val_idx]
    y_tr,  y_val  = y_train[tr_idx],     y_train[val_idx]

    fold_model = clone(grid_search.best_estimator_)
    fold_model.fit(X_tr, y_tr)

    y_pred  = fold_model.predict(X_val)
    y_proba = fold_model.predict_proba(X_val)[:, 1]

    m = compute_metrics(y_val, y_pred, y_proba)
    m['fold'] = fold + 1
    fold_metrics.append(m)

    print(f"Fold {fold+1} | "
          f"Acc: {m['accuracy']:.4f} | "
          f"Prec: {m['precision']:.4f} | "
          f"Rec: {m['recall']:.4f} | "
          f"F1: {m['f1']:.4f} | "
          f"AUROC: {m['auroc']:.4f}")

fold_df    = pd.DataFrame(fold_metrics).set_index('fold')
mean_row   = fold_df.mean().rename('mean')
std_row    = fold_df.std().rename('std')
cv_summary = pd.concat([fold_df, mean_row.to_frame().T, std_row.to_frame().T])

print('\n=== CV Results (best config) ===')
print(cv_summary.round(4))

wandb.log({
    'cv/accuracy_mean'  : fold_df['accuracy'].mean(),
    'cv/accuracy_std'   : fold_df['accuracy'].std(),
    'cv/precision_mean' : fold_df['precision'].mean(),
    'cv/precision_std'  : fold_df['precision'].std(),
    'cv/recall_mean'    : fold_df['recall'].mean(),
    'cv/recall_std'     : fold_df['recall'].std(),
    'cv/f1_mean'        : fold_df['f1'].mean(),
    'cv/f1_std'         : fold_df['f1'].std(),
    'cv/auroc_mean'     : fold_df['auroc'].mean(),
    'cv/auroc_std'      : fold_df['auroc'].std(),
    'cv_results'        : wandb.Table(dataframe=cv_summary.round(4))
})

## 10. Final Evaluation on Holdout Test Set (20%)
Best config already refit on full 80% train by GridSearchCV.
Evaluated exactly once on fixed holdout test set.

In [ ]:
y_pred_test  = grid_search.best_estimator_.predict(X_test_emb)
y_proba_test = grid_search.best_estimator_.predict_proba(X_test_emb)[:, 1]

test_metrics = compute_metrics(y_test, y_pred_test, y_proba_test)

# Save predictions for McNemar's test
np.save('y_pred-labse-finetuned-lr.npy', y_pred_test)
np.save('y_true-test.npy', y_test)

print('=== HOLDOUT TEST SET RESULTS ===')
print(f'  Accuracy  : {test_metrics["accuracy"]:.4f}')
print(f'  Precision : {test_metrics["precision"]:.4f}')
print(f'  Recall    : {test_metrics["recall"]:.4f}')
print(f'  F1        : {test_metrics["f1"]:.4f}')
print(f'  AUROC     : {test_metrics["auroc"]:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_test, target_names=le.classes_))

wandb.log({
    'test/accuracy'  : test_metrics['accuracy'],
    'test/precision' : test_metrics['precision'],
    'test/recall'    : test_metrics['recall'],
    'test/f1'        : test_metrics['f1'],
    'test/auroc'     : test_metrics['auroc']
})

report_df = pd.DataFrame(
    classification_report(y_test, y_pred_test,
                          target_names=le.classes_, output_dict=True)
).transpose().round(4)
wandb.log({'classification_report': wandb.Table(dataframe=report_df)})

## 11. Final Summary Table

In [ ]:
metrics_order = ['accuracy', 'precision', 'recall', 'f1', 'auroc']

summary = pd.DataFrame({
    'CV Mean' : fold_df[metrics_order].mean().round(4),
    'CV Std'  : fold_df[metrics_order].std().round(4),
    'Test'    : pd.Series(test_metrics)[metrics_order].round(4)
})

print('=== PAPER TABLE - Logistic Regression (LaBSE Finetuned) ===')
print(summary)
print('\nBest hyperparameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')

wandb.log({'paper_table': wandb.Table(dataframe=summary)})
wandb.config.update({
    **{f'best_param/{k}': v for k, v in grid_search.best_params_.items()}
})
wandb.finish()